# 🔬 Mamba-PFAN Cholec80 Scan-Direction Ablation Study

This notebook reproduces the complete **Mamba-PFAN scan-direction ablation matrix**, counts generator parameters, trains the 4 directional variants on Kaggle GPU, and computes evaluation metrics (PSNR, SSIM, MSE, and CIEDE2000).

### ⚙️ Kaggle Setup Checklist:
1. **Accelerator**: Select **GPU T4 x1** or **GPU P100** (*Notebook options* -> *Accelerator*).
2. **Internet**: Toggle **Internet ON** (*Notebook options* -> *Settings* -> *Internet* -> **ON**).
3. **Data**: Add your Cholec80 desmoking dataset via **+ Add Input** (mounts under `/kaggle/input`).


In [ ]:
# Preflight Checks & Configuration
from pathlib import Path
import os
import sys
import urllib.request
import torch

# 1. Check GPU Accelerator
gpu_available = torch.cuda.is_available()
if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU Detected: {device_name}")
    GPU_IDS = '0'
    DEVICE = 'cuda'
else:
    print("⚠️ WARNING: GPU is NOT detected! Training will be slow on CPU.")
    print("👉 In Kaggle sidebar: Notebook options -> Accelerator -> Select 'GPU T4 x1' or 'GPU P100'.")
    GPU_IDS = '-1'
    DEVICE = 'cpu'

# 2. Check Internet Connectivity
def check_internet():
    try:
        urllib.request.urlopen('https://github.com', timeout=5)
        return True
    except Exception:
        return False

internet_on = check_internet()
if internet_on:
    print("✅ Internet connection: ON")
else:
    print("⚠️ WARNING: Internet is OFF! Toggle Internet ON in Kaggle Notebook Settings if cloning from GitHub.")

# 3. Path Configuration
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('./input')
WORK_ROOT = Path('/kaggle/working/mamba_pfan')
CHECKPOINT_ROOT = WORK_ROOT / 'checkpoints'
RESULT_ROOT = WORK_ROOT / 'results'
REPO_URL = 'https://github.com/GladwinDaniel/Desmoking.git'
REPO_BRANCH = 'main'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# 4. Training Hyperparameters
# TIP: For a quick smoke test, set EPOCHS = 1. For full benchmark, set EPOCHS = 50.
EPOCHS = 50
BATCH_SIZE = 4
PRINT_FREQ = 20           # Prints loss every 20 batches (80 images)
SKIP_EXISTING = True      # Skips modes whose latest_net_G.pth already exists

# The 4 ablation variants:
MODES = [
    'horizontal_forward',        # 1-way horizontal unidirectional
    'horizontal_bidirectional',  # 2-way horizontal bidirectional
    'vertical_bidirectional',    # 2-way vertical bidirectional
    'four_way'                   # 4-way full cross-directional SSM
]

print(f"Work directory: {WORK_ROOT}")
print(f"Checkpoints directory: {CHECKPOINT_ROOT}")
print(f"Results directory: {RESULT_ROOT}")
if INPUT_ROOT.exists():
    print("Available input items:", [p.name for p in INPUT_ROOT.iterdir()])


In [ ]:
# Install Dependencies & Clone/Update Repository
import subprocess
import shutil

# Install required dependencies
%pip install -q timm einops scikit-image Pillow numpy

REPO_ROOT = WORK_ROOT / 'mamba_repo'

# Check if repo was uploaded directly as a Kaggle dataset
found_uploaded_repo = None
if INPUT_ROOT.exists():
    for candidate in INPUT_ROOT.rglob('mamba_repo*'):
        if candidate.is_dir() and (candidate / 'train.py').is_file():
            found_uploaded_repo = candidate
            break

if found_uploaded_repo:
    print(f"Found uploaded repository at {found_uploaded_repo}. Copying to {REPO_ROOT}...")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(found_uploaded_repo, REPO_ROOT)
elif (REPO_ROOT / '.git').is_dir():
    print(f"Repository already cloned at {REPO_ROOT}. Fetching latest updates from {REPO_BRANCH}...")
    try:
        subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=str(REPO_ROOT), check=True)
        subprocess.run(['git', 'reset', '--hard', f'origin/{REPO_BRANCH}'], cwd=str(REPO_ROOT), check=True)
        print("Updated repository to latest commit.")
    except Exception as e:
        print(f"Git update failed ({e}). Re-cloning fresh...")
        shutil.rmtree(REPO_ROOT)
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
            check=True
        )
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH}) into {REPO_ROOT}...")
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
        check=True
    )
    print("Cloned successfully.")

# Validate required ablation files
required_files = ['train.py', 'run_scan_ablation.py', 'evaluate_models.py', 'report_parameters.py']
for name in required_files:
    if not (REPO_ROOT / name).is_file():
        raise FileNotFoundError(f"Missing required file '{name}' in {REPO_ROOT}")

# Add REPO_ROOT to sys.path and set IPython working directory
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%cd {REPO_ROOT}
print(f"✅ Current working directory: {Path.cwd()}")


In [ ]:
# Robust Dataset Discovery & Split Verification
import zipfile
from PIL import Image

def find_dataset_folders(root_dir):
    """Scan for folders containing train / test data without traversing every file."""
    root = Path(root_dir)
    found_train = None
    found_test = None
    
    for current, dirs, _ in os.walk(root):
        cur_path = Path(current)
        # Check for train composite or trainA+trainB
        if found_train is None:
            if (cur_path / 'train').is_dir():
                found_train = cur_path
            elif (cur_path / 'trainA').is_dir() and (cur_path / 'trainB').is_dir():
                found_train = cur_path
        # Check for test folder
        if found_test is None:
            if (cur_path / 'test').is_dir():
                found_test = cur_path / 'test'
            elif (cur_path / 'testA').is_dir() and (cur_path / 'testB').is_dir():
                found_test = cur_path
            elif (cur_path / 'val').is_dir():
                found_test = cur_path / 'val'
        if found_train and found_test:
            break
            
    return found_train, found_test

# 1. Search in INPUT_ROOT
DATA_ROOT, TEST_ROOT = find_dataset_folders(INPUT_ROOT)

# 2. If not found in extracted format, check for uploaded ZIP files
if DATA_ROOT is None:
    zip_candidates = list(INPUT_ROOT.rglob('*.zip'))
    zip_candidates = [z for z in zip_candidates if 'mamba_repo' not in z.name.lower()]
    if zip_candidates:
        extraction_dir = Path('/kaggle/working/extracted_dataset')
        print(f"Extracting dataset archive {zip_candidates[0]} to {extraction_dir}...")
        extraction_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(extraction_dir)
        DATA_ROOT, TEST_ROOT = find_dataset_folders(extraction_dir)

if DATA_ROOT is None:
    raise FileNotFoundError(
        f"Could not find dataset under {INPUT_ROOT}. "
        "Expected folders: 'train' or 'trainA' + 'trainB'."
    )

print(f"✅ Dataset root: {DATA_ROOT}")

# 3. Ensure paired training images exist for aligned dataset mode
train_composite_dir = DATA_ROOT / 'train'
if not train_composite_dir.is_dir() or not any(train_composite_dir.glob('*.*')):
    train_a = DATA_ROOT / 'trainA'
    train_b = DATA_ROOT / 'trainB'
    if train_a.is_dir() and train_b.is_dir():
        print("Detected separate trainA and trainB. Generating composite side-by-side images for training...")
        aligned_root = Path('/kaggle/working/aligned_dataset')
        aligned_train = aligned_root / 'train'
        aligned_train.mkdir(parents=True, exist_ok=True)
        valid_exts = {'.bmp', '.png', '.jpg', '.jpeg'}
        a_files = sorted([f for f in train_a.iterdir() if f.suffix.lower() in valid_exts])
        for a_path in a_files:
            b_path = train_b / a_path.name
            if b_path.is_file():
                im_a = Image.open(a_path).convert('RGB')
                im_b = Image.open(b_path).convert('RGB')
                comp = Image.new('RGB', (im_a.width + im_b.width, max(im_a.height, im_b.height)))
                comp.paste(im_a, (0, 0))
                comp.paste(im_b, (im_a.width, 0))
                comp.save(aligned_train / a_path.name)
        print(f"Generated {len(list(aligned_train.glob('*')))} composite training images in {aligned_train}")
        DATA_ROOT = aligned_root

# 4. Resolve and verify paired test images
if TEST_ROOT is None:
    if (DATA_ROOT / 'test').is_dir():
        TEST_ROOT = DATA_ROOT / 'test'
    elif (DATA_ROOT / 'testA').is_dir() and (DATA_ROOT / 'testB').is_dir():
        TEST_ROOT = DATA_ROOT
    elif (DATA_ROOT / 'val').is_dir():
        TEST_ROOT = DATA_ROOT / 'val'

print(f"✅ Test root: {TEST_ROOT}")

# Test pair verification using evaluate_models resolver
from evaluate_models import resolve_pairs
test_pairs = resolve_pairs(TEST_ROOT)
print(f"✅ Successfully verified {len(test_pairs)} test image pairs!")
print("Sample test pairs:", [p[0] for p in test_pairs[:5]])


## 📊 Parameter Count Report
Counts generator parameters for both PFAN and Mamba-PFAN under identical network settings.


In [ ]:
# Run Parameter Report
import json

param_report_path = WORK_ROOT / 'parameter_report.json'
!python "{REPO_ROOT}/report_parameters.py" --output "{param_report_path}"

if param_report_path.is_file():
    report_data = json.loads(param_report_path.read_text())
    print("\n--- Model Parameter Comparison ---")
    for m in report_data['models']:
        print(f"• {m['model'].upper()}: {m['total_parameters']:,} parameters ({m['total_millions']:.3f}M)")
    if 'published_to_mamba_ratio' in report_data:
        print(f"Published PFAN to Mamba-PFAN ratio: {report_data['published_to_mamba_ratio']:.2f}x")


## 🚀 Train the Four Scan-Direction Variants
Trains the ablation models:
1. `horizontal_forward`: 1-way horizontal unidirectional SSM
2. `horizontal_bidirectional`: 2-way horizontal bidirectional SSM
3. `vertical_bidirectional`: 2-way vertical bidirectional SSM
4. `four_way`: 4-way cross-scan bidirectional SSM

*Note: With `SKIP_EXISTING = True`, models that have already finished training will not be repeated if this cell is rerun.*


In [ ]:
# Execute Scan Ablation Training
modes_arg = " ".join(MODES)
skip_flag = "--skip-existing" if SKIP_EXISTING else ""

cmd = (
    f'python "{REPO_ROOT}/run_scan_ablation.py" '
    f'--dataroot "{DATA_ROOT}" '
    f'--checkpoints-dir "{CHECKPOINT_ROOT}" '
    f'--name mamba_scan '
    f'--modes {modes_arg} '
    f'--n-epochs {EPOCHS} '
    f'--batch-size {BATCH_SIZE} '
    f'--print-freq {PRINT_FREQ} '
    f'--gpu-ids {GPU_IDS} '
    f'{skip_flag} '
    f'--manifest-output "{WORK_ROOT}/scan_ablation_commands.json"'
)

print(f"Executing: {cmd}\n")
!{cmd}


In [ ]:
# Comprehensive Evaluation & Summary Comparison Table
import json
import pandas as pd

eval_records = []

for mode in MODES:
    checkpoint = CHECKPOINT_ROOT / f'mamba_scan_{mode}' / 'latest_net_G.pth'
    output_dir = RESULT_ROOT / mode
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if not checkpoint.is_file():
        print(f"⚠️ Checkpoint not found for mode '{mode}': {checkpoint}")
        continue
        
    print(f"\nEvaluating scan mode: {mode}...")
    !python "{REPO_ROOT}/evaluate_models.py" \
        --model mamba_pfan \
        --scan-mode {mode} \
        --checkpoint "{checkpoint}" \
        --test-dir "{TEST_ROOT}" \
        --output-dir "{output_dir}" \
        --device {DEVICE}
        
    summary_file = output_dir / 'evaluation_summary.json'
    if summary_file.is_file():
        summary = json.loads(summary_file.read_text())
        metrics = summary['metrics']
        eval_records.append({
            'Scan Mode': mode,
            'PSNR (dB)': f"{metrics['psnr_db']['mean']:.2f} ± {metrics['psnr_db']['std']:.2f}",
            'SSIM': f"{metrics['ssim']['mean']:.4f} ± {metrics['ssim']['std']:.4f}",
            'CIEDE2000': f"{metrics['ciede2000']['mean']:.2f} ± {metrics['ciede2000']['std']:.2f}",
            'MSE': f"{metrics['mse']['mean']:.2f} ± {metrics['mse']['std']:.2f}",
            'raw_psnr': metrics['psnr_db']['mean'],
            'raw_ssim': metrics['ssim']['mean'],
        })

# Display formatted comparison table
if eval_records:
    df_eval = pd.DataFrame(eval_records)
    print("\n" + "=" * 65)
    print("📊 SCAN-DIRECTION ABLATION STUDY RESULTS (Cholec80)")
    print("=" * 65)
    display_cols = ['Scan Mode', 'PSNR (dB)', 'SSIM', 'CIEDE2000', 'MSE']
    print(df_eval[display_cols].to_string(index=False))
    
    # Save overall summary CSV
    summary_csv_path = RESULT_ROOT / 'ablation_summary.csv'
    df_eval[display_cols].to_csv(summary_csv_path, index=False)
    print(f"\nSaved ablation summary to {summary_csv_path}")


In [ ]:
# Visual Comparison Figure Across Ablation Variants
import matplotlib.pyplot as plt
from PIL import Image

if 'test_pairs' in locals() and len(test_pairs) > 0:
    sample_name, sample_in_path, sample_tgt_path = test_pairs[0]
    
    num_cols = len(MODES) + 2
    fig, axes = plt.subplots(1, num_cols, figsize=(3.5 * num_cols, 4))
    
    # Input Image
    axes[0].imshow(Image.open(sample_in_path).convert('RGB'))
    axes[0].set_title("Input (Hazy)", fontsize=10, fontweight='bold')
    axes[0].axis('off')
    
    # Mode outputs
    for idx, mode in enumerate(MODES):
        out_img_path = RESULT_ROOT / mode / f"{Path(sample_name).stem}_output.png"
        ax = axes[idx + 1]
        if out_img_path.is_file():
            ax.imshow(Image.open(out_img_path).convert('RGB'))
        else:
            ax.text(0.5, 0.5, "N/A", ha='center', va='center')
        ax.set_title(mode.replace('_', ' ').title(), fontsize=10)
        ax.axis('off')
        
    # Ground Truth Image
    axes[-1].imshow(Image.open(sample_tgt_path).convert('RGB'))
    axes[-1].set_title("Ground Truth (Clean)", fontsize=10, fontweight='bold')
    axes[-1].axis('off')
    
    plt.tight_layout()
    comparison_fig_path = RESULT_ROOT / 'ablation_visual_comparison.png'
    plt.savefig(comparison_fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved visual comparison figure to {comparison_fig_path}")


In [ ]:
# Package Results for Download
import shutil

export_dir = Path('/kaggle/working/export_bundle')
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Copy results (metrics, figures, sample predictions)
if RESULT_ROOT.exists():
    shutil.copytree(RESULT_ROOT, export_dir / 'results')

# 2. Copy parameter report & manifests
for file_name in ['parameter_report.json', 'scan_ablation_commands.json']:
    src = WORK_ROOT / file_name
    if src.is_file():
        shutil.copy(src, export_dir / file_name)

# 3. Copy generator checkpoints
checkpoints_export = export_dir / 'checkpoints'
checkpoints_export.mkdir(parents=True, exist_ok=True)
for mode in MODES:
    mode_ckpt_dir = CHECKPOINT_ROOT / f'mamba_scan_{mode}'
    if mode_ckpt_dir.is_dir():
        dest_mode = checkpoints_export / f'mamba_scan_{mode}'
        dest_mode.mkdir(parents=True, exist_ok=True)
        for ckpt_file in mode_ckpt_dir.glob('*net_G.pth'):
            shutil.copy(ckpt_file, dest_mode / ckpt_file.name)

# 4. Create ZIP archive
zip_output_path = '/kaggle/working/mamba_pfan_ablation_results'
archive_file = shutil.make_archive(zip_output_path, 'zip', export_dir)
archive_size_mb = Path(archive_file).stat().st_size / (1024 * 1024)

print("=" * 60)
print(f"🎉 Results successfully packaged!")
print(f"📦 Archive: {archive_file} ({archive_size_mb:.2f} MB)")
print("👉 You can download this file from Kaggle's right-hand Output panel.")
print("=" * 60)
